# U-Net for Lung Segmentation and Preprocessing

### Introduction
U-Net is a convolutional neural network architecture developed for fast and precise **biomedical image segmentation**. Unlike standard classification models that output a single label per image, U-Net performs pixel-wise prediction to create a mask of the region of interest.

**Project Role & Constraints:**
*   **Role:** This module is intended for **lung-region extraction** and image preprocessing.
*   **Ensemble Status:** This notebook is **not** a direct member of the classification soft-voting ensemble (which includes DenseNet, EfficientNet, and Swin-style models).
*   **Utility:** The segmentation output can be used to crop or mask images to focus on lung tissue, potentially reducing noise for downstream classification tasks.

> "This notebook is not a direct member of the classification soft-voting ensemble. It can instead be used as a preprocessing step before classification."

In [7]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt

# --- Dataset Configuration Placeholder ---
# Replace these paths with your actual paired segmentation data
IMAGE_DIR = 'path/to/chest_xray_images/'
MASK_DIR = 'path/to/lung_masks/'
IMG_SIZE = (256, 256)
CHANNELS = 1  # Standard for grayscale X-rays; change to 3 for RGB if needed

def load_segmentation_data():
    """
    Template logic for loading paired images and masks.
    Full training requires a dataset of (image, mask) pairs.
    """
    print("Status: Segmentation dataset not yet linked.")
    print("To proceed, ensure IMAGE_DIR and MASK_DIR contain paired files.")
    # In practice, this would return (X_train, y_train, X_val, y_val)
    return None, None, None, None

X_train, y_train, X_val, y_val = load_segmentation_data()

Status: Segmentation dataset not yet linked.
To proceed, ensure IMAGE_DIR and MASK_DIR contain paired files.


### Model Definition
We define a robust U-Net architecture featuring a contracting path (encoder), an expansive path (decoder), and skip connections to preserve spatial information.

In [8]:
def conv_block(inputs, filters):
    x = layers.Conv2D(filters, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def encoder_block(inputs, filters):
    x = conv_block(inputs, filters)
    p = layers.MaxPooling2D((2, 2))(x)
    return x, p

def decoder_block(inputs, skip_features, filters):
    x = layers.Conv2DTranspose(filters, (2, 2), strides=2, padding='same')(inputs)
    x = layers.Concatenate()([x, skip_features])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(256, 256, 1)):
    inputs = layers.Input(input_shape)

    # Encoder (Contracting Path)
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)

    # Bottleneck
    b1 = conv_block(p3, 512)

    # Decoder (Expansive Path)
    d1 = decoder_block(b1, s3, 256)
    d2 = decoder_block(d1, s2, 128)
    d3 = decoder_block(d2, s1, 64)

    # Output layer (Sigmoid for binary mask probability)
    outputs = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(d3)

    model = models.Model(inputs, outputs, name='U-Net_Lung_Seg')
    return model

model = build_unet(input_shape=(IMG_SIZE[0], IMG_SIZE[1], CHANNELS))
model.summary()

Model: "U-Net_Lung_Seg"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 256, 256,  │        640 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_15[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_14       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 256, 256,  │     36,928 │ activation_14[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_16[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_15       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 128, 128,  │          0 │ activation_15[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 128, 128,  │     73,856 │ max_pooling2d_3[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_17[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_16       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 128, 128,  │    147,584 │ activation_16[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_18[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_17       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 64, 64,    │          0 │ activation_17[0]… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 64, 64,    │    295,168 │ max_pooling2d_4[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │      1,024 │ conv2d_19[0][0] 

 Total params: 7,707,457 (29.40 MB)

 Trainable params: 7,701,825 (29.38 MB)

 Non-trainable params: 5,632 (22.00 KB)

### Training and Validation
In this section, we train the model on image-mask pairs.

In [9]:
### Segmentation Metrics and Compilation

def iou(y_true, y_pred):
    def f(y_true, y_pred):
        intersection = (y_true * y_pred).sum()
        union = y_true.sum() + y_pred.sum() - intersection
        return (intersection + 1e-15) / (union + 1e-15)
    return tf.numpy_function(f, [y_true, y_pred], tf.float32)

def dice_coef(y_true, y_pred):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + 1e-15) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + 1e-15)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy', # BCE + Dice Loss could be an alternative for future improvements
    metrics=[dice_coef, iou, 'accuracy']
)

# --- Training Section (Template) ---
"""
# To execute training, provide paired data to load_segmentation_data()
callbacks_list = [
    callbacks.ModelCheckpoint('unet_lung_seg.h5', save_best_only=True),
    callbacks.EarlyStopping(monitor='val_dice_coef', mode='max', patience=5, restore_best_weights=True)
]

if X_train is not None:
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=16,
        callbacks=callbacks_list
    )
"""

"\n# To execute training, provide paired data to load_segmentation_data()\ncallbacks_list = [\n    callbacks.ModelCheckpoint('unet_lung_seg.h5', save_best_only=True),\n    callbacks.EarlyStopping(monitor='val_dice_coef', mode='max', patience=5, restore_best_weights=True)\n]\n\nif X_train is not None:\n    history = model.fit(\n        X_train, y_train,\n        validation_data=(X_val, y_val),\n        epochs=20,\n        batch_size=16,\n        callbacks=callbacks_list\n    )\n"

### Segmentation Metrics
We use **Dice Coefficient** and **Intersection over Union (IoU)** to evaluate the overlap between predicted masks and ground truth.

In [10]:
def dice_coefficient(y_true, y_pred, smooth=1):
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

# Add these to model evaluation or custom training loops

### Sample Prediction Visualization
Visualizing the original image, the ground truth mask, and the U-Net prediction.

In [11]:
def visualize_results(image, mask, prediction):
    """
    Presentation-ready visualization function for comparing ground truth and predictions.
    """
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1); plt.title('Original Image'); plt.imshow(image, cmap='gray')
    plt.subplot(1, 3, 2); plt.title('Ground Truth Mask'); plt.imshow(mask, cmap='gray')
    plt.subplot(1, 3, 3); plt.title('U-Net Prediction'); plt.imshow(prediction, cmap='gray')
    plt.show()

### Expected Workflow & Conclusion

**Intended Pipeline:**
1.  **Input:** Load raw Chest X-ray.
2.  **Segmentation:** Generate lung mask using this U-Net model.
3.  **Processing:** Apply mask to original image or crop to mask boundaries.
4.  **Classification:** Pass processed (cleaned) image to DenseNet/EfficientNet ensemble.

**Final Note:**
This notebook serves as a structured segmentation module. While full training requires paired segmentation masks, the architecture and evaluation pipeline are ready for project integration. It remains an independent preprocessing utility and is not used as a member of the soft-voting classification ensemble.